# INT8 Post-Training_quantization (PTQ)
**Project:** Evaluating Neural Network Quantization for Efficient Deep Learning Inference

**Goal:** Quantize a pretrained ResNet-18 model to INT8 without additional training, then compare its predictive performance, serialized model size, and CPU inference latency against the FP32 baseline.

**Measurements:** Test accuracy, serialized model size, and CPU inference latency.

**Calculations:** Accuracy change, size reduction, and latency improvement, all compared to FP32 baseline.

**Dataset:** CIFAR-10

**Model:** Pretrained ResNet-18 fine-tuned for CIFAR-10 (SamAdamDay/resnet18_cifar10).

**Evaluation size:** The same fixed 1000 test images used in the FP32 baseline.


# Configuring the Colab notebook
Note: Select **CPU** for runtime if available

In [1]:
import platform
import torch
import torchvision

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU") # Use CPU if available

Python: 3.13.16
PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
CUDA available: False
Using CPU


# Installing the module-loading library

In [2]:
%pip install -q timm # Loading pretrained image-classification model from Hugging Face

import copy
import os
import random
import time
import json
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete.")

Setup complete.


#Setting up TorchAO for quantization

In [3]:
import torch
import torch.ao.quantization as aq
from torch.ao.quantization.quantize_fx import (
    prepare_fx,
    convert_fx,
)

print("PyTorch:", torch.__version__)
print("Supported quantization engines:",
      torch.backends.quantized.supported_engines)

PyTorch: 2.11.0+cpu
Supported quantization engines: ['qnnpack', 'onednn', 'x86', 'fbgemm']


# Loading the pre-trained ResNet-18 model

In [4]:
MODEL_ID = "hf_hub:SamAdamDay/resnet18_cifar10"

model = timm.create_model(
    MODEL_ID,
    pretrained=True
)

model.eval()

print("Model loaded successfully.")
print("Architecture:", model.__class__.__name__)
print("Number of parameters:",
      sum(p.numel() for p in model.parameters()))
print("Number of output classes:", model.num_classes)

Model loaded successfully.
Architecture: ResNet
Number of parameters: 11181642
Number of output classes: 10


# Loading CIFAR-10 dataset for the ResNet-18
Note: The cell below might take approx. 25 minutes when first ran.

In [5]:
# Loading CIFAR-10 dataset for the ResNet-18 model
from torch.utils.data import DataLoader, Subset

DATA_DIR = "./data"
BATCH_SIZE = 25
EVAL_SIZE = 1000

# Using the preprocessing specified by the model card
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.49139968, 0.48215827, 0.44653124),
        std=(0.24703233, 0.24348505, 0.26158768)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

# Fixed and reproducible subset of the official test split
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(
    len(test_dataset), generator=generator
)[:EVAL_SIZE].tolist()

test_subset = Subset(test_dataset, indices)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Training images available:", len(train_dataset))
print("Official test images:", len(test_dataset))
print("Images in this evaluation:", len(test_subset))

Training images available: 50000
Official test images: 10000
Images in this evaluation: 1000


# Preparing the model for quantization
Note: The pretrained FP32 ResNet-18 model is copied to the CPU and prepared for static INT8 post-training-quantization (PTQ). The original model remains unchanged so that its predictions and performance can be compared with the quantized model.

In [12]:
import copy

# Preserving the original FP32 model
model_fp32_cpu = copy.deepcopy(model).cpu().eval()

# Creating a separate model for INT8 PTQ
model_to_quantize = copy.deepcopy(model_fp32_cpu).eval()

# Selecting a supported CPU quantization backend
supported_engines = torch.backends.quantized.supported_engines

if "x86" in supported_engines:
    backend = "x86"
elif "fbgemm" in supported_engines:
    backend = "fbgemm"
else:
    raise RuntimeError(
        f"No supported x86 quantization backend: {supported_engines}"
    )

torch.backends.quantized.engine = backend

print("Selected quantization backend:", backend)

# Get one real batch to define the input shape
example_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=True,
)

example_images, _ = next(iter(example_loader))
example_inputs = (example_images.cpu(),)

# Configuring static INT8 quantization
qconfig_mapping = aq.get_default_qconfig_mapping(backend)

# Inserting observers into supported model operations
prepared_model = prepare_fx(
    model_to_quantize,
    qconfig_mapping,
    example_inputs,
)

print("Model prepared for FX static PTQ.")
print(prepared_model.graph)

Selected quantization backend: x86


/tmp/ipykernel_28886/3965799836.py:41: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_model = prepare_fx(
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super()._

Model prepared for FX static PTQ.
graph():
    %x : torch.Tensor [num_users=1] = placeholder[target=x]
    %activation_post_process_0 : [num_users=1] = call_module[target=activation_post_process_0](args = (%x,), kwargs = {})
    %conv1 : [num_users=1] = call_module[target=conv1](args = (%activation_post_process_0,), kwargs = {})
    %activation_post_process_1 : [num_users=1] = call_module[target=activation_post_process_1](args = (%conv1,), kwargs = {})
    %maxpool : [num_users=1] = call_module[target=maxpool](args = (%activation_post_process_1,), kwargs = {})
    %activation_post_process_2 : [num_users=2] = call_module[target=activation_post_process_2](args = (%maxpool,), kwargs = {})
    %layer1_0_conv1 : [num_users=1] = call_module[target=layer1.0.conv1](args = (%activation_post_process_2,), kwargs = {})
    %activation_post_process_3 : [num_users=1] = call_module[target=activation_post_process_3](args = (%layer1_0_conv1,), kwargs = {})
    %layer1_0_drop_block : [num_users=1] = cal

# PTQ calibration

In [7]:
CALIBRATION_SIZE = 1000

calibration_indices = torch.randperm(
    len(train_dataset),
    generator=torch.Generator().manual_seed(SEED),
)[:CALIBRATION_SIZE].tolist()

calibration_dataset = Subset(
    train_dataset,
    calibration_indices,
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

# Collecting activation statistics from training images
prepared_model.eval()

with torch.inference_mode():
    for images, _ in calibration_loader:
        prepared_model(images.cpu())

print(
    f"Calibration completed on {len(calibration_dataset)} images."
)

Calibration completed on 1000 images.


# Converting the model

In [8]:
# Converting the calibrated model to a quantized model
quantized_model = convert_fx(prepared_model)
quantized_model.eval()

print("INT8 PTQ conversion completed.")
print(quantized_model)

/tmp/ipykernel_28886/271777960.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = convert_fx(prepared_model)


INT8 PTQ conversion completed.
GraphModule(
  (conv1): QuantizedConvReLU2d(3, 64, kernel_size=(7, 7), stride=(2, 2), scale=0.15728707611560822, zero_point=0, padding=(3, 3))
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Module(
    (0): Module(
      (conv1): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.13486850261688232, zero_point=70, padding=(1, 1))
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.10838102549314499, zero_point=66, padding=(1, 1))
    )
    (1): Module(
      (conv1): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.127720445394516, zero_point=65, padding=(1, 1))
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.11777372658252716, zero_poin

# Testing a batch to make sure compiled model runs

In [9]:
images, labels = next(iter(test_loader))
images = images.cpu()

with torch.inference_mode():
    outputs = quantized_model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Output dtype:", outputs.dtype)
print("Test batch executed successfully.")

Input shape: torch.Size([25, 3, 256, 256])
Output shape: torch.Size([25, 10])
Output dtype: torch.float32
Test batch executed successfully.


#Verifying that the quantization actually happened

In [10]:
from collections import Counter

quantized_modules = [
    module
    for module in quantized_model.modules()
    if "quantized" in type(module).__module__.lower()
]

print("Quantized module count:", len(quantized_modules))
print(
    "Quantized module types:",
    Counter(type(module).__name__ for module in quantized_modules),
)

assert len(quantized_modules) > 0, (
    "No quantized modules found. Do not record this as an INT8 result."
)

Quantized module count: 22
Quantized module types: Counter({'Conv2d': 19, 'ConvReLU2d': 1, 'Linear': 1, 'LinearPackedParams': 1})


#Comparing FP32 and INT8 image predictions

In [14]:
# Comparing FP32 and INT8 on the exact same test subset
model_fp32_cpu.eval()
quantized_model.eval()

fp32_correct = 0
int8_correct = 0
matching_predictions = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.cpu()
        labels = labels.cpu()

        fp32_outputs = model_fp32_cpu(images)
        int8_outputs = quantized_model(images)

        fp32_preds = fp32_outputs.argmax(dim=1)
        int8_preds = int8_outputs.argmax(dim=1)

        fp32_correct += (fp32_preds == labels).sum().item()
        int8_correct += (int8_preds == labels).sum().item()
        matching_predictions += (
            fp32_preds == int8_preds
        ).sum().item()
        total += labels.size(0)

print(f"FP32 accuracy: {100 * fp32_correct / total:.2f}%")
print(f"INT8 accuracy: {100 * int8_correct / total:.2f}%")
print(
    "Prediction agreement: "
    f"{100 * matching_predictions / total:.2f}%"
)
print(
    f"Images with different predictions: "
    f"{total - matching_predictions} / {total}"
)

FP32 accuracy: 92.4000%
INT8 accuracy: 92.4000%
Prediction agreement: 99.5000%
Images with different predictions: 5 / 1000


# Evaluating quantized (INT8) accuracy
Note: Before we evaluate INT8 accuracy, we will first need to import results from FP32 baseline so that we can compare the results afterwards. I have stored the file on my drive.

In [11]:
# Importing the FP32 baseline data from the json created in earlier notebook
import json

from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/My Drive/Colab Notebooks/fp32_baseline.json", "r") as f:
    baseline_data = json.load(f)

if isinstance(baseline_data, list):
    baseline_data = baseline_data[0]

baseline_accuracy = baseline_data["accuracy_percent"]


# Now we start to evaluate the quantized accuracy

correct = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.cpu()
        labels = labels.cpu()

        outputs = quantized_model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

int8_accuracy = 100.0 * correct / total

print(f"INT8 PTQ accuracy: {int8_accuracy:.2f}%")
print(f"FP32 baseline accuracy: {baseline_accuracy:.2f}%")
print(
    f"Accuracy change: "
    f"{int8_accuracy - baseline_accuracy:+.2f} percentage points"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


INT8 PTQ accuracy: 92.40%
FP32 baseline accuracy: 92.40%
Accuracy change: +0.00 percentage points


#Measuring model's file size

In [15]:
import io
import torch

# Measuring serialized state-dict size in memory
buffer = io.BytesIO()
torch.save(quantized_model.state_dict(), buffer)

int8_size_bytes = buffer.getbuffer().nbytes
int8_size_mib = int8_size_bytes / (1024 ** 2)

print(f"INT8 serialized size: {int8_size_mib:.2f} MiB")
print(f"INT8 serialized size: {int8_size_bytes:,} bytes")

# Comparing against the existing FP32 baseline
fp32_size_mib = baseline_data["serialized_size_mib"]

print(f"FP32 serialized size: {fp32_size_mib:.2f} MiB")
print(
    f"Size reduction: "
    f"{100 * (1 - int8_size_mib / fp32_size_mib):.2f}%"
)

INT8 serialized size: 10.79 MiB
INT8 serialized size: 11,313,115 bytes
FP32 serialized size: 42.73 MiB
Size reduction: 74.75%


#Measuring CPU inference latency time

In [17]:
import time
import statistics
from torch.utils.data import DataLoader, Subset

# Match the FP32 baseline benchmark settings
BENCHMARK_IMAGES = 256
BENCHMARK_BATCH_SIZE = 32
WARMUP_BATCHES = 2
REPEATS = 3

benchmark_subset = Subset(
    test_subset,
    list(range(min(BENCHMARK_IMAGES, len(test_subset))))
)

benchmark_loader = DataLoader(
    benchmark_subset,
    batch_size=BENCHMARK_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

def benchmark_cpu_latency(model, loader, warmup_batches=2, repeats=3):
    model.eval()
    timings = []

    # Warming up the model before measuring
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= warmup_batches:
                break
            model(images.cpu())

        # Repeat the timed run to reduce measurement noise
        for _ in range(repeats):
            total_images = 0
            start = time.perf_counter()

            for images, _ in loader:
                images = images.cpu()
                model(images)
                total_images += images.size(0)

            elapsed = time.perf_counter() - start
            timings.append(elapsed * 1000 / total_images)

    return {
        "mean_ms_per_image": statistics.mean(timings),
        "std_ms_per_image": (
            statistics.stdev(timings) if len(timings) > 1 else 0.0
        ),
        "all_runs_ms_per_image": timings,
    }

int8_latency = benchmark_cpu_latency(
    quantized_model,
    benchmark_loader,
    warmup_batches=WARMUP_BATCHES,
    repeats=REPEATS,
)

print(
    f"INT8 CPU latency: "
    f"{int8_latency['mean_ms_per_image']:.3f} ms/image"
)
print(
    f"Run-to-run standard deviation: "
    f"{int8_latency['std_ms_per_image']:.3f} ms/image"
)
print("Individual runs:", int8_latency["all_runs_ms_per_image"])

INT8 CPU latency: 63.962 ms/image
Run-to-run standard deviation: 0.060 ms/image
Individual runs: [63.909128121096614, 64.0275555937464, 63.95011085937341]


#Comparing metrics of FP32 vs. INT8

In [20]:
fp32_latency = baseline_data["cpu_latency_ms_per_image"]
fp32_size = baseline_data["serialized_size_mib"]

size_reduction_percent = 100 * (1 - int8_size_mib / fp32_size)
latency_change_percent = 100 * (
    int8_latency["mean_ms_per_image"] / fp32_latency - 1
)

print("=== FP32 vs INT8 PTQ ===")
print(f"FP32 accuracy: {baseline_accuracy:.2f}%")
print(f"INT8 accuracy: {int8_accuracy:.2f}%")
print(f"Accuracy change: {int8_accuracy - baseline_accuracy:+.2f} pp")
print(f"FP32 size: {fp32_size:.2f} MiB")
print(f"INT8 size: {int8_size_mib:.2f} MiB")
print(f"Size reduction: {size_reduction_percent:.2f}%")
print(f"FP32 latency: {fp32_latency:.3f} ms/image")
print(
    f"INT8 latency: "
    f"{int8_latency['mean_ms_per_image']:.3f} ms/image"
)
print(f"Latency change: {latency_change_percent:+.2f}%*")
print("*Note that negative means INT8 is faster. Positive means FP32 is faster.")

=== FP32 vs INT8 PTQ ===
FP32 accuracy: 92.40%
INT8 accuracy: 92.40%
Accuracy change: +0.00 pp
FP32 size: 42.73 MiB
INT8 size: 10.79 MiB
Size reduction: 74.75%
FP32 latency: 66.810 ms/image
INT8 latency: 63.962 ms/image
Latency change: -4.26%*
*Note that negative means INT8 is faster. Positive means FP32 is faster.


#Saving the quantized results as a JSON file

In [29]:
int8_results = {
    "model": MODEL_ID,
    "dataset": "CIFAR-10",
    "method": "FX graph-mode static INT8 PTQ",
    "seed": SEED,
    "evaluation_images": EVAL_SIZE,
    "accuracy_percent": int8_accuracy,
    "prediction_agreement_percent": 100 * matching_predictions / total,
    "different_predictions": total - matching_predictions,
    "serialized_size_mib": int8_size_mib,
    "cpu_latency_ms_per_image": int8_latency["mean_ms_per_image"],
    "cpu_latency_std_ms_per_image": int8_latency["std_ms_per_image"],
    "benchmark_images": BENCHMARK_IMAGES,
    "benchmark_batch_size": BENCHMARK_BATCH_SIZE,
    "latency_repeats": REPEATS,
    "quantization_backend": backend,
    "pytorch_version": torch.__version__,
}

with open("int8_ptq.json", "w") as f:
    json.dump(int8_results, f, indent=2)

print(json.dumps(int8_results, indent=2))

{
  "model": "hf_hub:SamAdamDay/resnet18_cifar10",
  "dataset": "CIFAR-10",
  "method": "FX graph-mode static INT8 PTQ",
  "seed": 42,
  "evaluation_images": 1000,
  "accuracy_percent": 92.4,
  "prediction_agreement_percent": 99.5,
  "different_predictions": 5,
  "serialized_size_mib": 10.789027214050293,
  "cpu_latency_ms_per_image": 63.96226485807214,
  "cpu_latency_std_ms_per_image": 0.06014196814895937,
  "benchmark_images": 256,
  "benchmark_batch_size": 32,
  "latency_repeats": 3,
  "quantization_backend": "x86",
  "pytorch_version": "2.11.0+cpu"
}


#Saving the JSON file to Google Drive

In [30]:
import os
import shutil

drive_folder = "/content/drive/My Drive/Colab Notebooks"
os.makedirs(drive_folder, exist_ok=True)

drive_path = os.path.join(drive_folder, "int8_ptq.json")
shutil.copy2("int8_ptq.json", drive_path)

print("Results saved to:", drive_path)

Results saved to: /content/drive/My Drive/Colab Notebooks/int8_ptq.json


#Saving the JSON file by downloading it to your computer